# House Price Prediction - Model Training

Train machine learning models on Supabase data to predict Vietnamese house prices.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import joblib
import pickle

sys.path.insert(0, str(Path.cwd().parent.parent))
from pipeline.supabase_handler import fetch_csv_from_supabase

print("✅ Imports successful")

## Step 1: Load Data from Supabase

In [ ]:
print("Loading data from Supabase...")
df = fetch_csv_from_supabase("Raw_Features")

# Convert price from VND to billion VND
df['price_billion_vnd'] = df['price_vnd'] / 1e9

print(f"✓ Loaded {len(df)} records")
print(f"\nPrice range: {df['price_billion_vnd'].min():.2f} - {df['price_billion_vnd'].max():.2f} billion VND")

## Step 2: Prepare Features

In [ ]:
# Define features and target
FEATURE_COLS = [
    'nearest_school_km', 'school_count_3km',
    'nearest_hospital_km', 'hospital_count_5km',
    'nearest_marketplace_km', 'marketplace_count_3km',
    'nearest_supermarket_km', 'supermarket_count_3km',
    'nearest_mall_km', 'mall_count_3km',
    'nearest_bus_stop_km', 'bus_stop_count_1km',
    'nearest_metro_km', 'metro_count_5km',
    'area_m2', 'distance_to_center_km',
    'locality_population_density'
]
TARGET_COL = 'price_billion_vnd'

print(f"Features: {len(FEATURE_COLS)}")
print(f"Target: {TARGET_COL}")

In [ ]:
# Prepare data
print("\nPreparing data...")

# Drop rows with missing prices
df_clean = df.dropna(subset=[TARGET_COL]).copy()
print(f"After dropping NaN prices: {len(df_clean)} records")

# Select features and target
X = df_clean[FEATURE_COLS].copy()
y = df_clean[TARGET_COL].copy()
print(f"Initial shape: X={X.shape}, y={y.shape}")

# Fill missing values
X = X.fillna(X.median())
print(f"After filling NaN: missing values={X.isnull().sum().sum()}")

# Remove outliers (price > 100B)
mask = y < 100
X = X[mask]
y = y[mask]
print(f"After removing outliers: {len(X)} records")

print(f"\n✓ Ready for training: X={X.shape}, y={y.shape}")

## Step 3: Split Data

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train set: {len(X_train)} samples")
print(f"Test set: {len(X_test)} samples")

## Step 4: Train Models

In [ ]:
# Random Forest
print("Training Random Forest...")
rf = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)
rf_r2 = r2_score(y_test, rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
rf_mae = mean_absolute_error(y_test, rf_pred)

print(f"✓ Random Forest")
print(f"  R²: {rf_r2:.4f}")
print(f"  RMSE: {rf_rmse:.4f} billion VND")
print(f"  MAE: {rf_mae:.4f} billion VND")

In [ ]:
# Gradient Boosting
print("\nTraining Gradient Boosting...")
gb = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
gb.fit(X_train, y_train)

gb_pred = gb.predict(X_test)
gb_r2 = r2_score(y_test, gb_pred)
gb_rmse = np.sqrt(mean_squared_error(y_test, gb_pred))
gb_mae = mean_absolute_error(y_test, gb_pred)

print(f"✓ Gradient Boosting")
print(f"  R²: {gb_r2:.4f}")
print(f"  RMSE: {gb_rmse:.4f} billion VND")
print(f"  MAE: {gb_mae:.4f} billion VND")

## Step 5: Select Best Model

In [ ]:
# Compare
comparison = pd.DataFrame({
    'Model': ['Random Forest', 'Gradient Boosting'],
    'R²': [rf_r2, gb_r2],
    'RMSE': [rf_rmse, gb_rmse],
    'MAE': [rf_mae, gb_mae]
})

print("Model Comparison:")
print(comparison.to_string(index=False))

# Select best
best_name = 'random_forest' if rf_r2 > gb_r2 else 'gradient_boosting'
best_model = rf if rf_r2 > gb_r2 else gb
best_r2 = max(rf_r2, gb_r2)

print(f"\n✅ Best Model: {best_name.upper()} (R² = {best_r2:.4f})")

## Step 6: Visualize Results

In [ ]:
# Get predictions from best model
y_best_pred = best_model.predict(X_test)
errors = y_test - y_best_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Actual vs Predicted
axes[0].scatter(y_test, y_best_pred, alpha=0.5)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[0].set_xlabel('Actual Price (billion VND)')
axes[0].set_ylabel('Predicted Price (billion VND)')
axes[0].set_title('Actual vs Predicted')
axes[0].grid(True, alpha=0.3)

# Error distribution
axes[1].hist(errors, bins=30, edgecolor='black', alpha=0.7)
axes[1].axvline(errors.mean(), color='r', linestyle='--', label=f'Mean: {errors.mean():.2f}')
axes[1].set_xlabel('Prediction Error (billion VND)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Error Distribution')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 7: Save Model

In [ ]:
# Save model
model_dir = Path('saved_models')
model_dir.mkdir(exist_ok=True)

model_path = model_dir / f"{best_name}.joblib"
joblib.dump(best_model, model_path)
print(f"✓ Saved model: {model_path}")

# Save metadata
metadata = {
    'model_type': best_name,
    'features': FEATURE_COLS,
    'metrics': {
        'r2_score': float(best_r2),
        'rmse': float(np.sqrt(mean_squared_error(y_test, y_best_pred))),
        'mae': float(mean_absolute_error(y_test, y_best_pred))
    },
    'train_size': len(X_train),
    'test_size': len(X_test)
}

meta_path = model_dir / f"{best_name}_meta.pkl"
with open(meta_path, 'wb') as f:
    pickle.dump(metadata, f)
print(f"✓ Saved metadata: {meta_path}")

print(f"\n✅ Model training complete!")